# Developpement des algorithmes ICA

Ce notebook est un brouillon

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproductibilite
rng = np.random.default_rng(42)

In [13]:
#génération de données synthétiques
D = 3  # nombre de sources
N = 5000  # nombre d'échantillons
S = np.zeros((D, N))
S[0] = rng.laplace(loc=0, scale=1, size=N)  # source 1: Laplace
S[1] = rng.uniform(low=-1, high=1, size=N)  # source 2: Uniforme centrée
S[2] = rng.standard_t(df=3, size=N)  # source 3: Student
A = rng.standard_normal((D, D))  # matrice de mélange aléatoire
X = A @ S  # données mélangées  


## Pretraitement (centrage + whitening)

On centre les donnees puis on applique un whitening PCA afin d'avoir une covariance proche de l'identite.
Ce pretraitement stabilise les mises a jour ICA.

In [19]:
# fonction de pretraitement

def center_whiten(X, eps=1e-8):
    mu = X.mean(axis=1, keepdims=True)
    Xc = X - mu

    cov = (Xc @ Xc.T) / Xc.shape[1]
    eigvals, eigvecs = np.linalg.eigh(cov)

    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    D_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals + eps))
    W_white = D_inv_sqrt @ eigvecs.T
    Xw = W_white @ Xc

    return Xw, mu, W_white

In [14]:
# Test rapide du pretraitement sur un melange synthetique

X = A @ S
Xw, mu, W_white = center_whiten(X)

cov_w = (Xw @ Xw.T) / N
err_cov = np.max(np.abs(cov_w - np.eye(D)))

print("Covariance blanchie (approx):")
print(np.round(cov_w, 3))
print(f"ecart max a I: {err_cov:.3e}")

assert err_cov < 5e-2, "Whitening insuffisant: covariance trop eloignee de I"
print("Pretraitement valide.")

Covariance blanchie (approx):
[[ 1. -0. -0.]
 [-0.  1.  0.]
 [-0.  0.  1.]]
ecart max a I: 9.423e-09
Pretraitement valide.


## FastICA 

On utilise `sklearn.decomposition.FastICA`
On evalue la separation avec l'indice d'Amari sur $C = \hat V A$.

In [20]:
from sklearn.decomposition import FastICA

def amari_index(C):
    D = C.shape[0]
    absC = np.abs(C)

    row_max = absC.max(axis=1, keepdims=True)
    col_max = absC.max(axis=0, keepdims=True)

    row_term = (absC / (row_max + 1e-12)).sum(axis=1) - 1.0
    col_term = (absC / (col_max + 1e-12)).sum(axis=0) - 1.0

    return (row_term.sum() + col_term.sum()) / (2.0 * D * (D - 1))

fastica = FastICA(
    n_components=D,
    algorithm='parallel',
    fun='logcosh',
    whiten='unit-variance',
    max_iter=1000,
    tol=1e-6,
    random_state=42,
    )





In [21]:
S_hat = fastica.fit_transform(X.T).T
V_hat = fastica.components_

C_fastica = V_hat @ A
amari_fastica = amari_index(C_fastica)
print(f"Amari FastICA sklearn: {amari_fastica:.4f}")
print("Matrice C = V_hat A (arrondie):")
print(np.round(C_fastica, 3))


Amari FastICA sklearn: 0.0129
Matrice C = V_hat A (arrondie):
[[ 5.000e-03  1.737e+00 -2.000e-03]
 [-7.030e-01  3.200e-02 -1.000e-03]
 [-9.000e-03  2.000e-02  5.780e-01]]


## SGD-ICA

## Adam-ICA

## Evaluation (indice d'Amari)

## Comparaison et visualisation